### Middleware 
Middleware provides a way to more tightly control what happens inside the agents. Middleware is useful for the following:
- Tracking agent behavior with logging , analytics and debugging.
- Transforming prompts , tool selection and output formatting.
- Adding retries , fallbacks and ealry termination logic.
- Applying rate limits , guradrails and PII detection.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["Gemini_API_Key"] = os.getenv("Gemini_API_Key")

## Summarization Middleware 
- Automatically summarizes conversation history when approching token limits, preserving recent messge while compressing older context . Summarization is usefull for the following:
- Long-runing conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.


In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver 
from langchain_core.messages import HumanMessage , AIMessage , SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["Gemini_API_Key"] = os.getenv("Gemini_API_Key")



model = ChatGoogleGenerativeAI(
      model="gemini-2.5-flash",
      google_api_key=os.getenv("Gemini_API_Key")
)

agent = create_agent(
    model = model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)


In [26]:
## Run with thread id
config = {"configurable":{"thread_id":"test-1"}}

In [27]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?"
    "what is 15-7?",
    "what is 4*4?"
    "What is 16*8?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages :{response}")
    print(f"Message:{len(response['messages'])}")

Messages :{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='3942dd26-1c2d-420e-821e-c5ab6f182f2f'), AIMessage(content='2+2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ef842-b994-7583-868a-ca55724647c6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 24, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 18}})]}
Message:2
Messages :{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='3942dd26-1c2d-420e-821e-c5ab6f182f2f'), AIMessage(content='2+2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ef842-b994-7583-868a-ca55724647c6-0', tool_calls=[

In [ ]:

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver 
from langchain_core.messages import HumanMessage , AIMessage , SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

@tool
def search_hotels(city:str) ->str:
    """Search Hotels - returns long response to use more token"""
    return f"""Hotels in {city}
1. Grand Hotal - 5 star , $350/night , spa , pool , gym
2. City Inn - 4 star , $180/night , business center 
3. Budget Stay - 3 star , $75/night , free wifi 
"""

agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",10),
            keep = ("tokens",10)
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars

In [5]:
cities = ["Paris","London","New York","Dubai"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens , {len(response["messages"])} messages")
    print(f"{(response["messages"])}")

Paris: ~993 tokens , 4 messages
[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user's primary goal is to find hotels in Paris.\n\n## SUMMARY\n\nThe user initiated the conversation with a request to find hotels in Paris. No further details, choices, or strategies have been discussed yet.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nThe next step is to begin searching for hotels in Paris. This may involve asking the user for additional criteria (e.g., dates, price range, specific amenities, number of guests) to refine the search.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='cde1f1a0-0cd1-4bbf-ad79-e775bc2fb0e5'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'5a956323-46fb-4d47-a77c-954bd2d772f0': 'CrICAQw51sfs86VAolkVM2FfsCoNVOzHltO0XiBKK7qZ8yx5SKJfu87sacbDBP4KeaA7Q0zaCyz1Cq3VNPZZfsWMKO

### Human in Loop Middleware 
Pause agent exection for human approval ,editing or rejection of tool calls before they execute . Human-in-the -loop is useful for the following.
- High-stackes operation requireing Humna approval (e.g databse writes , financial transactions).
- Compliances workfolws where Human Oversight is manadatory.
- Long-ruuning conversations where human feedback guides the agent.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agenet = create_agent(
    model =  model,
    tools=[read_email_tool , send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
)
    ]
)




In [17]:
config = {"configurable": {"thread_id": "test-approve"}}

result = agenet.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

In [18]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7349b173-a6f6-4e39-9a7c-5f0714a730c2'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "subject": "hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'9bc40325-15ae-43ef-8c36-d88adf2c0486': 'Cp0EAQw51sdpxfJCNPPkDlexehlnU5eSgFM+NXRHUX2vJ2Cz8mAxYg4ZSsQK8aFntc/BUwpBYYyLY5tIPWAbaMlutoYgYm9gLF536LczR44BDLxDKKs2hvGfrbBO40t2rz1v4R47d2V8QDM2riBxXqabuTK108hUwT2KGv5MD+0eKjhXkR7jxjJZBvaTub/uFHNtefq502xjMVP4pLFNsF2xvG0ZUFAj1zCjD/54uZMgGdf+sPYgauL/rypsLVaiiAnfELIKcE+wLwRv+xGPjcWDdO8hChRfS37nVB35NvPvUOaeVOHE00S/Q8g7+mLbBXgUa8+S5VW2twAUb64pPGpj8E6aei5fMK1lKSdZmBKy5JqK5fMakwwPGvfnyiQBbCMPp4FiDPwQHxkk1OVxXTgZFyh7XEzpenFRDtrU9cJ8Qu89kVwgDtbs/qbCZdQ5NorFQpP9oalMf6JplwpaqNRNxJVD7GZRoOztYxSfcE6jGdQX1/JBnUPX3Xd0ojX8RZbUKQmNQrGySPURfW03

In [20]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused Approving...")

    result = agenet.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

Paused Approving...


In [21]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7349b173-a6f6-4e39-9a7c-5f0714a730c2'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "subject": "hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'9bc40325-15ae-43ef-8c36-d88adf2c0486': 'Cp0EAQw51sdpxfJCNPPkDlexehlnU5eSgFM+NXRHUX2vJ2Cz8mAxYg4ZSsQK8aFntc/BUwpBYYyLY5tIPWAbaMlutoYgYm9gLF536LczR44BDLxDKKs2hvGfrbBO40t2rz1v4R47d2V8QDM2riBxXqabuTK108hUwT2KGv5MD+0eKjhXkR7jxjJZBvaTub/uFHNtefq502xjMVP4pLFNsF2xvG0ZUFAj1zCjD/54uZMgGdf+sPYgauL/rypsLVaiiAnfELIKcE+wLwRv+xGPjcWDdO8hChRfS37nVB35NvPvUOaeVOHE00S/Q8g7+mLbBXgUa8+S5VW2twAUb64pPGpj8E6aei5fMK1lKSdZmBKy5JqK5fMakwwPGvfnyiQBbCMPp4FiDPwQHxkk1OVxXTgZFyh7XEzpenFRDtrU9cJ8Qu89kVwgDtbs/qbCZdQ5NorFQpP9oalMf6JplwpaqNRNxJVD7GZRoOztYxSfcE6jGdQX1/JBnUPX3Xd0ojX8RZbUKQmNQrGySPURfW03

In [30]:
## Reject

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agenet = create_agent(
    model =  model,
    tools=[read_email_tool , send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
)
    ]
)

In [31]:
config = {"configurable": {"thread_id": "test-approve"}}

result = agenet.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

In [32]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused Approving...")

    result = agenet.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

Paused Approving...


In [33]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='273fd879-3dc5-4f3c-8aca-5b3c9e547b5a'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "subject": "hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'e44890ae-1737-4cba-9180-5e28ec4fdf9e': 'CpcFAQw51sdPnqeQJbQt6dAUpalxCn9g9x93q3mVONsugSzipEdR9fJIj2BKYY9MTxdqk8Y1gJMeJr/7kH4Gr/ZtxJMzjuup0YqDaRvNqn47z8065odOXVt+3rtRHISWUj0NGUCrqSsFeofcdqfGfQ524KZL8L2HR1IApJcCmz65Ou2s8LfqAy/Wk9AxXqTTw+6WaW22xVnOR/Qn3J0Z+Jj+OyQOY2/MrZ5lFpShkwZNXUgYm0/mnpqBIcknM37aKHEj299cPbPD5G/9gf2pQwS79BZp6hY5YCOhIYN57BGxfPHQQwjbVbKBfu+0yNEk7RPx7ywMgLPuRx/b1zpd0DcU8zhfYhx/IOuDVljJDMCVwBjbhXQ5MoH11gmlS/2mjzbh8O3Gbh7IT7g7ZmTmPl0cZdBE/dJG8sWVN2hQYpczM6lwGmFXxloYZN+ySMCmmyhd3LqiY0C5cdySyajVz4ne1cp68Hlesha2b0hA5X/MBtT8D2nOYSYp6aEOvSpVbhgxSbv1yYeZzXb7WtM7

In [34]:
## Editing 

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agenet = create_agent(
    model =  model,
    tools=[read_email_tool , send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
)
    ]
)

In [36]:
config = {"configurable": {"thread_id": "test-approve"}}

result = agenet.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a0535d8f-bf4c-49df-8462-324219009deb'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "hello"}'}, '__gemini_function_call_thought_signatures__': {'919ca28c-58da-4b0e-8b68-469b02682201': 'CoIFAQw51sddRkE1lm3G0wDommW8dNHBovYRG9/1SLZ2nSEcsyXKhwUBOk4zTpqzqer+zuDWMRydUViDMu1z8eylN7tqbB8HM8ptB1lJ1xBHAi2HUMvW2FFVT0AJ8Vmm9nc1aCZGCtqgPvHhVrcPa7f0BzfaoXemMFiGvPHW8G+7NEE7jDAxjynrpTeDeQ/cmMGNbnAgQeVppiEAubC3WkSwk907Pujnv9mzqqyjp9smEMHjWoYq9fYYH/pcBmOUDMTR4cImDDaPEvm3MrOBF/Yc9MQOefkhZcPq54I+1X8NUkR0QIshNaLVPGi828+JF+HecDaxAHFhcU0BA4lhEgySib2XjmbX5SUGvg4FbGLDPspNAbIekTPzB7jPEh0gNCmXX2/Hq6v1ppnjkT792c8fKsgJngwaUY5XNLFAu8k00VbEg035G3uPfipZY676DsbAXiyyTp1dJk8oIf9kvewKbI3f8N59+c8wRVkOPsJYvGXqVWdC4UWbYXSI2XjZAb2jmclt57VzsUFAmuGc

In [37]:
if "__interrupt__" in result:
    print("Paused Editing...")

    result = agenet.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recipient":"correct@email.com",
                                "subject":"Corrected Subject",
                                "body":"This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
            
        ),
        config=config
    )

Paused Editing...


In [38]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a0535d8f-bf4c-49df-8462-324219009deb'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "recipient": "john@test.com", "subject": "hello"}'}, '__gemini_function_call_thought_signatures__': {'919ca28c-58da-4b0e-8b68-469b02682201': 'CoIFAQw51sddRkE1lm3G0wDommW8dNHBovYRG9/1SLZ2nSEcsyXKhwUBOk4zTpqzqer+zuDWMRydUViDMu1z8eylN7tqbB8HM8ptB1lJ1xBHAi2HUMvW2FFVT0AJ8Vmm9nc1aCZGCtqgPvHhVrcPa7f0BzfaoXemMFiGvPHW8G+7NEE7jDAxjynrpTeDeQ/cmMGNbnAgQeVppiEAubC3WkSwk907Pujnv9mzqqyjp9smEMHjWoYq9fYYH/pcBmOUDMTR4cImDDaPEvm3MrOBF/Yc9MQOefkhZcPq54I+1X8NUkR0QIshNaLVPGi828+JF+HecDaxAHFhcU0BA4lhEgySib2XjmbX5SUGvg4FbGLDPspNAbIekTPzB7jPEh0gNCmXX2/Hq6v1ppnjkT792c8fKsgJngwaUY5XNLFAu8k00VbEg035G3uPfipZY676DsbAXiyyTp1dJk8oIf9kvewKbI3f8N59+c8wRVkOPsJYvGXqVWdC4UWbYXSI2XjZAb2jmclt57VzsUFAmuGc